# Track 4 — Analítica, ML, IA y GenAI
**Rol:** CRB_ANALITICA | **Tiempo:** 15 min | **Criterio:** AI functions, agentes RAG, observabilidad, IA responsable

In [ ]:
USE ROLE CRB_ANALITICA;
USE DATABASE CREDIBANCO_HOL;
USE WAREHOUSE CREDIBANCO_HOL_WH;

## Bloque 1 — Ejecutar: AI nativa en SQL (no notebooks)

In [ ]:
-- AI_COMPLETE: extracción de campos desde texto no estructurado
SELECT documento_id,
  AI_COMPLETE('llama3.1-70b',
    'Del siguiente texto extrae: tipo de riesgo, entidad involucrada y monto. Responde en JSON: ' || TEXTO_DOCUMENTO
  )::VARCHAR AS extraccion
FROM CREDIBANCO_HOL.CUMPLIMIENTO.DOCUMENTOS_SARLAFT LIMIT 3;

In [ ]:
-- AI_CLASSIFY: clasificación de severidad con labels cortos
SELECT alerta_id, comercio_id, descripcion,
  AI_CLASSIFY(descripcion, ARRAY_CONSTRUCT('ALT','MED','BAJ')):labels[0]::VARCHAR AS severidad
FROM CREDIBANCO_HOL.RIESGO.ALERTAS LIMIT 5;

In [ ]:
-- Cortex Search: RAG sobre documentos SARLAFT (ya indexado)
SELECT PARSE_JSON(
  SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
    'CREDIBANCO_HOL.CUMPLIMIENTO.CSS_SARLAFT_DOCS_USER',
    '{"query": "lavado de activos", "columns": ["TEXTO_DOCUMENTO","TIPO"], "limit": 3}'
  )
);

## Bloque 2 — Evidencia: Anomalías y riesgo pre-computados

In [ ]:
-- Anomalías detectadas por comercio
SELECT * FROM CREDIBANCO_HOL.ANALITICA.RESULTADO_ANOMALIAS
WHERE IS_ANOMALY = TRUE LIMIT 10;

In [ ]:
-- Scoring de riesgo por comercio
SELECT ES_FRAUDE, COUNT(*) AS n, ROUND(AVG(TICKET_PROMEDIO),0) AS ticket_prom
FROM CREDIBANCO_HOL.ANALITICA.TRAIN_RIESGO_COMERCIO
GROUP BY 1;

## Bloque 3 — CoCo
Copia este prompt en Cortex Code:

> **Crea un agente RAG que combine el Cortex Search Service de documentos SARLAFT con la tabla de alertas de riesgo. El agente debe poder responder preguntas como '¿qué comercios tienen alertas de lavado?' cruzando documentos con datos transaccionales.**

In [ ]:
-- Verificación final
SELECT 'T4_COMPLETO' AS status,
  (SELECT COUNT(*) FROM CREDIBANCO_HOL.ANALITICA.RESULTADO_ANOMALIAS WHERE IS_ANOMALY = TRUE) AS anomalias;